In [1]:
import base64
import io
import json
import os
import wave
from pathlib import Path

import numpy as np
import sounddevice as sd
import websocket
from dotenv import load_dotenv
from IPython.display import Audio, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
REALTIME_MODEL = "gpt-realtime"
REALTIME_VOICE = "marin"
INPUT_WAV_PATH = "outputs/realtime_input.wav"  # 24kHz, mono, PCM16 WAV
OUTPUT_WAV_PATH = "outputs/openai_realtime_s2s.wav"



## OpenAI Realtime (Speech-to-Speech)

Reference: https://platform.openai.com/docs/guides/realtime-websocket

Notes:
- Input audio should be 24kHz, mono, PCM16 WAV to match `audio/pcm`.
- Voice is set on the session before generating audio output.


In [2]:
def read_pcm16_wav(path: str, sample_rate: int = 24000) -> bytes:
    with wave.open(path, "rb") as wf:
        channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        rate = wf.getframerate()
        if channels != 1 or sampwidth != 2 or rate != sample_rate:
            raise ValueError(
                f"Expected mono PCM16 {sample_rate}Hz WAV. Got channels={channels}, "
                f"sampwidth={sampwidth}, rate={rate}."
            )
        return wf.readframes(wf.getnframes())


def pcm16_to_wav_bytes(pcm: bytes, sample_rate: int = 24000, channels: int = 1) -> bytes:
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm)
    return buf.getvalue()


def record_wav(
    seconds: float = 5.0,
    sample_rate: int = 24000,
    output_path: str = INPUT_WAV_PATH,
):
    print(f"Recording {seconds}s at {sample_rate}Hz...")
    audio = sd.rec(int(seconds * sample_rate), samplerate=sample_rate, channels=1, dtype="float32")
    sd.wait()
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16)

    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm.tobytes())

    print(f"Saved {output_path}")



In [4]:
def openai_realtime_s2s(
    input_wav_path: str = INPUT_WAV_PATH,
    output_wav_path: str = OUTPUT_WAV_PATH,
    voice: str = REALTIME_VOICE,
    model: str = REALTIME_MODEL,
):
    if not OPENAI_API_KEY:
        raise ValueError("Missing OPENAI_API_KEY in notebooks/.env")

    pcm_in = read_pcm16_wav(input_wav_path)

    url = f"wss://api.openai.com/v1/realtime?model={model}"
    auth = OPENAI_API_KEY.strip().strip("'")
    if not auth.startswith("Bearer "):
        auth = f"Bearer {auth}"

    ws = websocket.create_connection(url, header=[f"Authorization: {auth}"])

    def recv_event():
        return json.loads(ws.recv())

    def wait_for_session_updated():
        while True:
            event = recv_event()
            etype = event.get("type")
            if etype == "session.updated":
                return event
            if etype == "error":
                raise RuntimeError(event.get("error"))

    # Consume initial session.created if present
    try:
        first = recv_event()
        if first.get("type") == "error":
            raise RuntimeError(first.get("error"))
    except Exception:
        pass

    session_update = {
        "type": "session.update",
        "session": {
            "type": "realtime",
            "model": model,
            "output_modalities": ["audio"],
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "turn_detection": None,
                },
                "output": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "voice": voice,
                },
            },
        },
    }
    ws.send(json.dumps(session_update))
    wait_for_session_updated()

    # Clear and send input audio
    ws.send(json.dumps({"type": "input_audio_buffer.clear"}))
    ws.send(
        json.dumps(
            {
                "type": "input_audio_buffer.append",
                "audio": base64.b64encode(pcm_in).decode("ascii"),
            }
        )
    )
    ws.send(json.dumps({"type": "input_audio_buffer.commit"}))

    # Explicit response audio format to avoid schema defaults
    ws.send(
        json.dumps(
            {
                "type": "response.create",
                "response": {
                    "output_modalities": ["audio"],
                    "audio": {"output": {"format": {"type": "audio/pcm", "rate": 24000}}},
                },
            }
        )
    )

    out_pcm = bytearray()
    transcript = None
    while True:
        event = recv_event()
        etype = event.get("type")
        if etype in ("response.output_audio.delta", "response.audio.delta"):
            out_pcm.extend(base64.b64decode(event.get("delta", "")))
        elif etype in ("response.output_audio_transcript.done", "response.audio_transcript.done"):
            transcript = event.get("transcript")
        elif etype == "response.done":
            break
        elif etype == "error":
            raise RuntimeError(event.get("error"))

    ws.close()

    wav_bytes = pcm16_to_wav_bytes(bytes(out_pcm))
    if output_wav_path:
        path = Path(output_wav_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(wav_bytes)

    return wav_bytes, transcript



In [5]:
record_wav(seconds=5)


Recording 5s at 24000Hz...
Saved outputs/realtime_input.wav


In [6]:
audio_bytes, transcript = openai_realtime_s2s()


In [7]:
display(Audio(audio_bytes))


In [8]:
transcript


'Hi Abhay! Great to meet you. I’m your friendly AI assistant, here to chat with you about anything and everything. We can definitely talk! How’s your day going so far?'